In [ ]:
import pickle
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import numpy as np
from tqdm import tqdm
# import df2img

In [ ]:

path = Path("/nfs/research/goldman/anoufa/data/dpca/storing_file.pkl")

with open(path, "rb") as f:
    storing_file = pickle.load(f)
    
median_storing_file = storing_file[0]
mean_storing_file = storing_file[1]
het_sites_storing_file = storing_file[2]
prop_below_depth_storing_file = storing_file[3]
length_storing_file = storing_file[4]
amplicon_length_storing_file = storing_file[5]
dropout_length_storing_file = storing_file[6]

In [ ]:
sum(median_storing_file), sum(length_storing_file), sum(mean_storing_file)
# Number of samples accessed

In [ ]:
# plot the five distributions
# (Files are already categorized counts)

median_cat = [f"{i*100:.0f} - {(i+1)*100 - 1:.0f}" for i in range(12)]

median_cat.append("1200+")

plt.figure(figsize=(5, 6))
plt.bar(median_cat, median_storing_file, color='#3B6FB6', alpha=0.9, label='Median')
plt.xticks(rotation=45)
# Display one xtick every 2 categories
plt.xticks(ticks=range(0, len(median_cat), 2), labels=median_cat[::2])
plt.ylabel("Number of samples")
plt.xlabel("Median coverage")
# plt.title("Distribution of median depth values in the dataset")
plt.tight_layout()
plt.savefig("../figs/median_depth_distribution.png")
plt.show()


In [ ]:

# ECDF plot using Plotly:

# Create a DataFrame for Plotly
all_data = []
counts = median_storing_file


for i, count in enumerate(counts):
    # Estimate a numeric midpoint for sorting and labeling
    if i < 12:
        start_val = i * 100
        end_val = (i + 1) * 100 - 1
        mid_val = (start_val + end_val) / 2
        label = f"{start_val} - {end_val}"
    else:
        mid_val = 1250  # just a representative value > 1200
        label = "1200+"
    
    all_data.append({
        "Median Depth Label": label,
        "Median Depth Order": mid_val,
        "Count": count
    })

df = pd.DataFrame(all_data)

# Sort and compute cumulative sum
df_sorted = df.sort_values(by="Median Depth Order")
df_sorted["Cumulative Count"] = df_sorted["Count"].cumsum()
df_sorted["Cumulative Prop"] = df_sorted["Cumulative Count"] / df_sorted["Cumulative Count"].max()

# Plot with Plotly
fig = px.line(
    df_sorted,
    x="Median Depth Label",
    y="Cumulative Prop",
    labels={
        "Median Depth Label": "Median coverage",
        "Cumulative Prop": "Cumulative proportion of samples"
    },
    markers=True
)

fig.update_layout(
    xaxis_tickangle=30,
    width=600,
    height=600,
    title_x=0.5,
    xaxis_title="Median coverage",
    yaxis_title="Cumulative proportion of samples",
    font=dict(color="black", size=13),
    xaxis=dict(tickfont=dict(size=13)),
    yaxis=dict(tickfont=dict(size=13))
)

# Remove the right frame and top frame
fig.update_layout(
    margin=dict(l=40, r=10, t=10, b=40),  # Adjust these as needed (left, right, top, bottom)
    autosize=True
)

# Save to HTML
fig.write_html("../figs/median_depth_cumulative_distribution_plotly.html")

In [ ]:
len(median_storing_file), len(mean_storing_file), len(het_sites_storing_file), len(prop_below_depth_storing_file), len(length_storing_file), len(amplicon_length_storing_file), len(dropout_length_storing_file)

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(median_cat, mean_storing_file, color='#3B6FB6', alpha=0.9, label='Median')
plt.xticks(rotation=45)
plt.ylabel("Counts")
plt.xlabel("Mean depth categories")
plt.title("Distribution of mean depth values in the dataset")
plt.tight_layout()
plt.savefig("../figs/mean_depth_distribution.png")
plt.show()

In [ ]:
s = sum(length_storing_file[1:])

print(f"Maple misses {s} samples because of their length exceeding 29903bp (insertions).")

In [ ]:
length_storing_file = length_storing_file[:np.max(np.nonzero(length_storing_file)) + 1]

length_cat = [f"{i+29903:.0f}" for i in range(len(length_storing_file) - 1)]

length_cat.append(f"{len(length_storing_file) + 29903 - 1:.0f}+")

plt.figure(figsize=(12, 6))
plt.bar(length_cat, length_storing_file, color='#D41645', alpha=0.8, label='Length')
plt.xticks(rotation=45)
plt.ylabel("Counts")
plt.xlabel("Length categories")
plt.title("Distribution of length values in the dataset")
plt.tight_layout()
plt.savefig("../figs/length_distribution.png")
plt.show()


In [ ]:
sum(het_sites_storing_file[3][:])

In [ ]:
# Cut het_sites_storing_file at the max index non-zero

het_sites_cat = [f"{i:.0f}" for i in range(5)]
het_sites_cat.append("5-9")
het_sites_cat.append("10-19")
het_sites_cat.append("20-29")
het_sites_cat.append("30+")

het_thrs = [0.7, 0.8, 0.9, 0.95]

plt.figure(figsize=(12, 6))
for i in range(len(het_sites_storing_file)):
    plt.subplot(2, 2, i+1)
    plt.bar(het_sites_cat, het_sites_storing_file[i], color='#F49E17', alpha=0.8, label='Heterozygous sites')
    plt.xticks(rotation=45)
    plt.ylabel("Counts")
    plt.xlabel("Number of heterozygous sites")
    plt.title(f"Distribution of number of heterozygous sites (consensus<{het_thrs[i]*100}%)")
plt.tight_layout()
plt.savefig("../figs/het_sites_distribution.png")
plt.show()


In [ ]:

# for i in range(len(prop_below_depth_storing_file)):
#     prop_below_depth_storing_file[i] = prop_below_depth_storing_file[i][:np.max(np.nonzero(prop_below_depth_storing_file[i])) + 1]


# depth_thrs = [0.1, 0.02]

# plt.figure(figsize=(12, 6))
# for i in range(len(prop_below_depth_storing_file)):
#     plt.subplot(1, 2, i+1)
#     prop_below_cat_v2 = [f"{i*0.05*29903:.0f} - {(i+1)*0.05*29903:.0f}" for i in range(len(prop_below_depth_storing_file[i]))]

#     plt.bar(prop_below_cat_v2, prop_below_depth_storing_file[i], color='#18974C', alpha=0.8, label='Proportion below depth')
#     plt.xticks(rotation=45)
#     plt.ylabel("Counts")
#     plt.xlabel("Proportion below depth categories")
#     plt.title(f"Distribution of the number of positions below {depth_thrs[i]*100:.0f}% median depth")

# plt.tight_layout()
# plt.savefig("../figs/prop_below_depth_distribution.png")
# plt.show()



In [ ]:
prop_below_depth_storing_file_reduced = [prop_below_depth_storing_file[i][:10] for i in range(len(prop_below_depth_storing_file))]

In [ ]:
# Make the two bar plots on the same graph using plotly

# Trim trailing zeros
for i in range(len(prop_below_depth_storing_file)):
    prop_below_depth_storing_file[i] = prop_below_depth_storing_file[i][:np.max(np.nonzero(prop_below_depth_storing_file[i])) + 1]

depth_thrs = [0.2, 0.1, 0.05, 0.02, 0.01]
all_data = []

# Prepare data for Plotly
for i in range(len(prop_below_depth_storing_file_reduced)):
    values = prop_below_depth_storing_file_reduced[i]
    categories = [f"{i*0.02*29903:.0f} - {(i+1)*0.02*29903:.0f} (<{(i+1)*0.02*100:.0f}%)" for i in range(len(values))]
    for cat, val in zip(categories, values):
        all_data.append({
            "Proportion Category": cat,
            "Count": val,
            "Threshold": f"< {depth_thrs[i]*100:.0f}%"
        })

df = pd.DataFrame(all_data)

# Plotly grouped bar chart
fig = px.bar(
    df,
    x="Proportion Category",
    y="Count",
    color="Threshold",
    color_discrete_sequence=['#193F90', '#8BB8E8', '#563D82', '#CBA3D8', '#18974C'],
    barmode="group",
    labels={"Proportion Category": "Proportion below depth categories"},
    # title="Distribution of the number of positions below thresholds of median depth"
)

fig.update_layout(xaxis_tickangle=45)
# Make it squared
fig.update_layout(
    width=600,
    height=600,
    title_x=0.5,
    xaxis_title="Proportion below depth categories",
    yaxis_title="Counts",
    legend_title_text="Thresholds"
)
# fig.write_image("../figs/prop_below_depth_distribution_plotly.png")
# fig.write_html("../figs/prop_below_depth_distribution_plotly.html")
fig.show()

In [ ]:
all_data = []

# Prepare data for Plotly, including a numerical order column for sorting
for i in range(len(prop_below_depth_storing_file_reduced)):
    values = prop_below_depth_storing_file_reduced[i]
    for j, val in enumerate(values): # Use j for index within values to get numerical category order
        start_val = j * 0.02 * 29903
        end_val = (j + 1) * 0.02 * 29903
        
        all_data.append({
            "Proportion Category_Label": f"<{end_val:.0f} / <{(j+1)*0.02*100:.0f}%", # String label for x-axis
            "Proportion Category_Order": start_val, # Numerical value for sorting
            "Count": val,
            "Threshold": f"<{depth_thrs[i]*100:.0f}%"
        })

df = pd.DataFrame(all_data)

# Sort the DataFrame by Threshold and then by the numerical order of the categories
df_sorted = df.sort_values(by=['Threshold', 'Proportion Category_Order'])

# Calculate the cumulative sum of 'Count' for each 'Threshold' group
df_sorted['Cumulative Count'] = df_sorted.groupby('Threshold')['Count'].cumsum()
df_sorted['Cumulative Prop'] = df_sorted['Cumulative Count']/df_sorted['Cumulative Count'].max()

# Plotly grouped bar chart for cumulative values
fig = px.line(
    df_sorted,
    x="Proportion Category_Label", # Use the label for display on x-axis
    y="Cumulative Prop", # Use the cumulative count for y-axis
    color="Threshold",
    color_discrete_sequence=['#193F90', '#CBA3D8', '#563D82', '#8BB8E8', '#18974C'],
    category_orders={"Threshold": ["<1%", "<2%", "<5%", "<10%", "<20%"]},
    labels={
        "Proportion Category_Label": "Number of sites below depth",
        "Cumulative Prop": "Cumulative prop" # Updated y-axis label
    },
    markers=True,
    # title="Cumulative Distribution of the number of positions below thresholds of median depth"
)

fig.update_layout(xaxis_tickangle=30)
fig.update_layout(
    width=600,
    height=600,
    title_x=0.5,
    xaxis_title="Number / Proportion of positions under the threshold",
    yaxis_title="Cumulative proportion of samples",
    legend_title_text="Thresholds"
)

# Use a darker font color and larger font size for better visibility
fig.update_layout(
    font=dict(
        color="black",
        size=13
    ),
    xaxis=dict(
        tickfont=dict(size=13)
    ),
    yaxis=dict(
        tickfont=dict(size=13)
    )
)

# Remove the right frame and top frame
fig.update_layout(
    margin=dict(l=40, r=20, t=10, b=40),  # Adjust these as needed (left, right, top, bottom)
    autosize=True
)

# Save the plot
output_html_path = "../figs/prop_below_depth_cumulative_distribution_plotly.html"

fig.write_html(output_html_path)

fig

In [ ]:
# # Concatenate all the n_masked files

# folder_path = Path("/nfs/research/goldman/anoufa/data/dpca/gen_metrics/")


# # Generate a new file to reset it
# n_masked_list_total = [[0] * 60 for _ in range(2)]

# for pkl_file in folder_path.glob("*.pkl"):
#     parameters = pkl_file.stem.split("_")
#     with open(pkl_file, "rb") as f:
#         storing_list = pickle.load(f)
    
#     for i in range(len(storing_list)):
#         n_masked_list_total[i] = [x + y for x, y in zip(n_masked_list_total[i], storing_list[i])]
        
#     # Remove the file
#     pkl_file.unlink()

# # Check index of non-zero values of both lists
# id_max_1 = np.max(np.nonzero(n_masked_list_total[0]))
# id_max_2 = np.max(np.nonzero(n_masked_list_total[1]))

# id_max = max(id_max_1, id_max_2)
# # Remove the rest of the list
# n_masked_list_total[0] = n_masked_list_total[0][:id_max + 1]
# n_masked_list_total[1] = n_masked_list_total[1][:id_max + 1]

# # Save the new file
# total_path = Path(f"/nfs/research/goldman/anoufa/data/dpca/n_masked_list_total_{parameters[3]}_{parameters[4]}.pkl")
# with open(total_path, "wb") as f:
#     pickle.dump(n_masked_list_total, f)
    

In [ ]:
med_depth = 0.1
cons_prop = 1.0

total_path = Path(f"/nfs/research/goldman/anoufa/data/dpca/n_masked_list_total_{cons_prop}_{med_depth}.pkl")

with open(total_path, "rb") as f:
    n_masked_list_total = pickle.load(f)
    
# Sum values over 7000 to have a category 7000+ (elem [14:])

n_masked_list_total[0] = n_masked_list_total[0][:14] + [sum(n_masked_list_total[0][14:])]
n_masked_list_total[1] = n_masked_list_total[1][:14] + [sum(n_masked_list_total[1][14:])]

list_cons_masked = n_masked_list_total[0]
list_masked = n_masked_list_total[1]

cat_masked = [f"{i*500} - {(i+1)*500 - 1}" for i in range(len(list_masked)-1)]
cat_masked.append("7000+")

print(len(list_masked), len(list_cons_masked), len(cat_masked))

# Plot the two barplots side by side using plotly
df = pd.DataFrame({
    'Category': cat_masked,
    'Masked': list_masked,
    'Viridian': list_cons_masked
})

fig = px.bar(df, x='Category', y=['Masked', 'Viridian'], barmode='group',
                title=f"Distribution of the number of masked sites when masking sites with depth < {med_depth*100:.0f}% median depth and consensus <= {cons_prop*100:.0f}%",
                labels={'value': 'Counts', 'Category': 'Masked Sites Categories'},
                color_discrete_sequence=['#0A5032', '#18974C'])
fig.update_layout(xaxis_title='Number of masked sites', yaxis_title='Counts')
fig.write_image(f"../figs/masked_sites_distribution_{med_depth}_{cons_prop}.png", engine="kaleido", validate=True)
fig.write_html(f"../figs/masked_sites_distribution_{med_depth}_{cons_prop}.html")
fig.show()

In [ ]:
sum(list_masked), sum(list_cons_masked)

In [ ]:
# # This script should go over the tsv results of MAPLE sample placement and
# # generate the distributions of the branch lengths for the three types of sequences
# # (viridian, masked, random).

# # The goal is to see whether the masked samples are closer to the tree than the viridian ones or not.

# import gzip
# viridian_lengths = [0] * 9
# masked_lengths = [0] * 9
# random_lengths = [0] * 9

# def bin_branch_lengths(length):
#     """
#     Bin the branch lengths into categories.
#     """
#     # One bin every 0.5 until 4+ (9 categories)
    
#     if length < 0.1:
#         return 0
#     elif length < 0.9:
#         return 1
#     elif length < 1.1:
#         return 2
#     elif length < 1.9:
#         return 3
#     elif length < 2.1:
#         return 4
#     elif length < 2.9:
#         return 5
#     elif length < 3.1:
#         return 6
#     elif length < 3.9:
#         return 7
#     else:
#         return 8
        

# # Define the path to the folder containing the tsv files
# file_path = Path("/nfs/research/goldman/anoufa/data/MAPLE_output/output_FULL_metaData_samplePlacements_1.0_0.1_0.15_100_3000.gz")
# unit = 1/29903

# # Iterate over the tsv files in the folder
# # Read the tsv file
# with gzip.open(file_path, 'rt') as file:
#     lines = file.readlines()

# # Extract the sample lengths for the three types of sequences
# for line in lines[1:]:
#     # Columns:  read_name	consensus	masked	    random
#     #           DRR287340	10.045929	10.054023	10.051963
    
#     # We are interested in the sampleBlengths of the placement with highest support.
#     columns = line.strip().split("\t")
#     sample = columns[0]
#     cons_blength = float(columns[1])
#     masked_blength = float(columns[2])
#     random_blength = float(columns[3])
    
#     # Bin the branch lengths
#     viridian_lengths[bin_branch_lengths(cons_blength)] += 1
#     masked_lengths[bin_branch_lengths(masked_blength)] += 1
#     random_lengths[bin_branch_lengths(random_blength)] += 1
    

# len_cat = ["0 - 0.1", "0.1 - 0.9", "0.9 - 1.1", "1.1 - 1.9",
#            "1.9 - 2.1", "2.1 - 2.9", "2.9 - 3.1", "3.1 - 3.9", "3.9+"]

# # Counts to percentages
# viridian_lengths = [x / sum(viridian_lengths) * 100 for x in viridian_lengths]
# masked_lengths = [x / sum(masked_lengths) * 100 for x in masked_lengths]
# random_lengths = [x / sum(random_lengths) * 100 for x in random_lengths]


# # Convert to long-form DataFrame
# df = pd.DataFrame({
#     "Category": len_cat * 3,
#     "Percentage": viridian_lengths + masked_lengths + random_lengths,
#     "Type": ["Viridian"] * len(len_cat) + ["Masked"] * len(len_cat) + ["Random"] * len(len_cat)
# })

# fig = px.bar(
#     df,
#     x="Category",
#     y="Percentage",
#     color="Type",
#     color_discrete_sequence=['#734595', '#CBA3D8', '#F4C61F'],
#     barmode="group",
#     title="Distribution of Branch lengths for Viridian, Masked, and Random Sequences",
#     labels={"Category": "Branch length with unit 1/29903"}
# )

# fig.update_layout(xaxis_tickangle=-45)
# fig.write_html("../figs/branch_lengths_distribution.html")
# fig.show()

In [ ]:
# Make the bar plot of the amplicon length distribution using plotly

# Trim trailing zeros
amplicon_length_storing_file = amplicon_length_storing_file[:np.max(np.nonzero(amplicon_length_storing_file)) + 1]

all_data = []

# Prepare data for Plotly
values = amplicon_length_storing_file
categories = [f"{i*100 + 50:.0f} - {(i+1)*100 + 50:.0f}" for i in range(len(values))]

for cat, val in zip(categories, values):
    all_data.append({
        "Length category": cat,
        "Count": val})

df = pd.DataFrame(all_data)

# Plotly grouped bar chart
fig = px.bar(
    df,
    x="Length category",
    y="Count",
    color_discrete_sequence=['#D41645'],
    barmode="group",
    title="Distribution of amplicon lengths in the dataset"
)

fig.update_layout(xaxis_tickangle=45)
fig.write_image("../figs/amplicon_length_dist_plotly.png")
fig.write_html("../figs/amplicon_length_dist_plotly.html")
fig.show()

In [ ]:
# Make the bar plot of the dropout length distribution using plotly

# Trim trailing zeros
dropout_length_storing_file = dropout_length_storing_file[:20]

all_data = []

# Prepare data for Plotly
values = dropout_length_storing_file
categories = [f"{i*50:.0f} - {(i+1)*50:.0f}" for i in range(len(values))]

for cat, val in zip(categories, values):
    all_data.append({
        "Length category": cat,
        "Count": val})

df = pd.DataFrame(all_data)

# Plotly grouped bar chart
fig = px.bar(
    df,
    x="Length category",
    y="Count",
    color_discrete_sequence=['#D41645'],
    barmode="group",
    title="Distribution of dropout lengths in the dataset (dropouts < 20x)"
)

fig.update_layout(xaxis_tickangle=45)
fig.write_image("../figs/dropout_length_dist_plotly.png")
fig.write_html("../figs/dropout_length_dist_plotly.html")
fig.show()

In [ ]:
# Evolution of the number of samples putatively contaminated detected depending on the threshold on the branch length difference.
# Threshold on max mut is fixed at 5

n_cont_samp = [335, 635, 2072, 16945]

thresholds = ["4", "3", "2", "1"]

import pandas as pd
import plotly.express as px

# Prepare data for Plotly
all_data = []
for i, val in enumerate(n_cont_samp):
    all_data.append({
        "Threshold": thresholds[i],
        "Number of contaminated samples": val
    })
    
df = pd.DataFrame(all_data)
# Plotly bar chart
fig = px.line(
    df,
    x="Threshold",
    y="Number of contaminated samples",
    markers=True,
    title="Evolution of the number of putatively contaminated samples detected",
    labels={"Threshold": "Threshold on branch length difference", "Number of contaminated samples": "Number of contaminated samples"},
    color_discrete_sequence=['#D41645']
)
fig.update_layout(xaxis_title="Threshold on branch length difference", yaxis_title="Number of contaminated samples")

# Make the fig squared
fig.update_layout(width=800, height=600)

fig.write_image("../figs/contaminated_samples_evolution.png")
fig.write_html("../figs/contaminated_samples_evolution.html")
fig.show()



### Comparing results with random

In [ ]:
# Expected random results

avg_n_masked = 863.2558093823226 - 388.61404963952157
genome_length = 29903
n_samples = 4119237
avg_cons_dist = 1.427409162736934
chance_of_masking_right_pos = avg_cons_dist / genome_length

# n_samples is the number of trials
# For each trial we mask avg_n_masked positions
# The probability of masking a position that makes a difference is chance_of_masking_right_pos

# We now compute the expected number of samples where we masked at least one right position
# This is a binomial distribution where:
# - n is the number of samples
# - p is the probability of masking at least one right position in a sample
# The probability of masking at least one right position in a sample is:

p = 1 - (1 - chance_of_masking_right_pos) ** avg_n_masked

expected_n_samples_masked = n_samples * p

expected_n_samples_masked, p

# This is much higher than what we find in fact (~20k), but at least it comforts us in the fact that the random results are not that outrageous.

In [ ]:
import pandas as pd

df_psmp = pd.read_csv("/nfs/research/goldman/anoufa/data/MAPLE_output/processed_placements_results_masked_1.0_0.1_1_0_30000.4.tsv", sep="\t")
df_psmp_random = pd.read_csv("/nfs/research/goldman/anoufa/data/MAPLE_output/processed_placements_results_random_1.0_0.1_1_0_30000.4.tsv", sep="\t")
# OR
# df_psmp_random = df_psmp.copy()

df_psmp.shape, df_psmp_random.shape
df_psmp['type'] = 'Masked'
df_psmp_random['type'] = 'Random'

df_psmp['dist_diff'] = df_psmp['consensus'] - df_psmp['masked']
df_psmp_random['dist_diff'] = df_psmp_random['consensus'] - df_psmp_random['random']

df_psmp_random['prop_dist_reduced'] = df_psmp_random['dist_diff'] / df_psmp_random['consensus']


In [ ]:
df_to_plot = df_psmp[['masked', 'type']].copy()
df_to_plot = pd.concat([df_to_plot, df_psmp_random[['random', 'type']].rename(columns={'random': 'masked'})], ignore_index=True)

df_to_plot.rename(columns={'masked': 'Masked/random branch length'}, inplace=True)

fig = px.histogram(
    df_to_plot,
    x='Masked/random branch length',
    color='type',
    barmode='overlay',
    title='Distribution of Masked/Random Branch Lengths',
    labels={'Masked/random branch length': 'Masked/Random Branch Length'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)

# fig.add_vline(x=20, line_dash='dash', line_color='black', annotation_text='Masked/Random Branch Length = 20', annotation_position='top right')
fig.update_yaxes(type='log')

fig.write_html("../figs/masked_random_branch_lengths_distribution.html")
fig.show()

In [ ]:
# Concatenate the two dataframes
df_psmp = pd.concat([df_psmp, df_psmp_random], ignore_index=True)

df_psmp.shape, df_psmp.groupby('type').size()

In [ ]:
df_psmp = df_psmp[df_psmp['consensus'] < 20]

df_psmp.groupby('type').size()

In [ ]:
df_psmp = df_psmp[df_psmp['consensus'] > 1.5]

df_psmp.groupby('type').size()

In [ ]:
# Plot prop_dist_reduced/prop_gen_masked dist

df_psmp['masking_ratio'] = df_psmp['prop_dist_reduced'] / df_psmp['prop_gen_masked']

# Set values over 50 to 50
df_psmp['masking_ratio'] = df_psmp['masking_ratio'].clip(upper=100)

# Lower clip at 0
df_psmp['masking_ratio'] = df_psmp['masking_ratio'].clip(lower=0)

# Plot the distribution of masking_ratio

fig = px.histogram(
    df_psmp,
    x='masking_ratio',
    nbins=50,
    color='type',
    barmode='overlay',
    labels={'masking_ratio': 'Masking Ratio'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)
fig.update_layout(xaxis_title='Masking Ratio (prop_dist_reduced / prop_gen_add_masked)', yaxis_title='Count')
fig.write_html("../figs/masking_ratio_distribution.html")
fig.show()

In [ ]:
# Plot dist_diff ecdf

# upper clip at 10
df_psmp['dist_diff'] = df_psmp['dist_diff'].clip(upper=10)


# Lower clip at 0
df_psmp['dist_diff'] = df_psmp['dist_diff'].clip(lower=0)

fig = px.ecdf(
    df_psmp,
    x='dist_diff',
    color='type',
    labels={'dist_diff': 'Distance Difference'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)
fig.update_layout(xaxis_title='Distance difference', yaxis_title='Cumulative proportion')
fig.update_traces(line_width=3) 

fig.update_layout(width=600, height=400)

# fig.add_vline(x=1, line_dash='dash', line_color='black', annotation_text='Distance diff = 1', annotation_position='top right')

# # Hatch 0 - 10 area
# fig.add_shape(
#     type="rect",
#     x0=0,
#     y0=0,
#     x1=1,
#     y1=1.1,
#     fillcolor="rgba(0, 0, 0, 0.1)",
#     line=dict(color="rgba(0, 0, 0, 0)"),
# )

fig.write_html("../figs/dist_diff_ecdf_full_dataset.html")
fig.show()

In [ ]:
fig = px.histogram(
    df_psmp,
    x='dist_diff',
    nbins=51,
    color='type',
    barmode='overlay',
    labels={'dist_diff': 'Distance Difference'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)

# Log scale on y-axis
fig.update_yaxes(type='log')
fig.update_layout(xaxis_title='Distance difference', yaxis_title='Count')
# fig.update_layout(width=600, height=400)

fig.write_html("../figs/dist_diff_distribution.html")
fig.show()

In [ ]:
# Same plot but as a cumulative proportion instead of count
fig = px.ecdf(
    df_psmp,
    x='masking_ratio',
    color='type',
    # nbins=50,
    # cumulative=True,
    # histnorm='probability',
    # title='Cumulative distribution of masking_ratio (prop_dist_reduced / prop_gen_masked)',
    labels={'masking_ratio': 'Masking Ratio'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)

fig.add_vline(x=10, line_dash='dash', line_color='red', annotation_text='Masking Ratio = 10', annotation_position='top right')

# Hatch 0 - 10 area
fig.add_shape(
    type="rect",
    x0=0,
    y0=0,
    x1=10,
    y1=1.1,
    fillcolor="rgba(255, 0, 0, 0.2)",
    line=dict(color="red", width=0),
)

fig.update_layout(xaxis_title='Masking Ratio', yaxis_title='Cumulative prop')
fig.update_traces(line_width=3) 

fig.update_layout(width=600, height=400)

fig.write_html("../figs/masking_ratio_cumulative_distribution.html")
fig.show()

In [ ]:
# Plot masked branch lengths dist

fig = px.ecdf(
    df_psmp,
    x='masked',
    color='type',
    title='Distribution of Masked Branch Lengths',
    labels={'masked': 'Masked Branch Length'},
    color_discrete_sequence=['#3B6FB6', '#734595']
)

fig.add_vline(x=20, line_dash='dash', line_color='black', annotation_text='Masked Branch Length = 20', annotation_position='top right')

# Hatch 20 - 100 area
fig.add_shape(
    type="rect",
    x0=20,
    y0=0,
    x1=100,
    y1=1,
    fillcolor="rgba(0, 0, 0, 0.2)",
    line=dict(color="black", width=0),
)

fig.update_layout(xaxis_title='Masked Branch Length', yaxis_title='Count')

fig.update_layout(width=800, height=400)

fig.write_html("../figs/masked_branch_lengths_distribution.html")
fig.show()

In [ ]:
# Plot difference between masked and consensus branch lengths

df_psmp['n_diff_mut'] = df_psmp['consensus'] - df_psmp['masked']

# upper clip at 10
# df_psmp['n_diff_mut'] = df_psmp['n_diff_mut'].clip(upper=10)

fig = px.ecdf(
    df_psmp,
    x='n_diff_mut',
    title='Distribution of difference between Masked and Consensus Branch Lengths',
    labels={'n_diff_mut': 'Difference between Masked and Consensus Branch Lengths'},
    color_discrete_sequence=['#F49E17']
)

fig.add_vline(x=2, line_dash='dash', line_color='black', annotation_text='Difference = 2', annotation_position='top right')

# Hatch 1 - 2
fig.add_shape(
    type="rect",
    x0=0,
    y0=0,
    x1=2,
    y1=1,
    fillcolor="rgba(0, 0, 0, 0.2)",
    line=dict(color="black", width=0),
)

fig.update_layout(xaxis_title='Difference between Masked and Consensus Branch Lengths', yaxis_title='Count')

# square shape fig
fig.update_layout(width=800, height=400)
fig.write_html("../figs/difference_masked_consensus_branch_lengths_distribution.html")
fig.show()

In [ ]:
# Plot difference between masked and consensus branch lengths

df_psmp['n_diff_mut_random'] = df_psmp['consensus'] - df_psmp['random']

# Plot n_diff_mut ecdf and n_diff_mut_random ecdf on the same figure

aux_df = df_psmp[['n_diff_mut', 'n_diff_mut_random']].melt(var_name='Type', value_name='Difference')

# upper clip at 10
aux_df['Difference'] = aux_df['Difference'].clip(upper=10)
# lower clip at 0
aux_df['Difference'] = aux_df['Difference'].clip(lower=-1)

type_labels = {
    'n_diff_mut': 'Masked vs Consensus',
    'n_diff_mut_random': 'Random vs Consensus'
}

# Apply the mapping to the 'Type' column
aux_df['Type'] = aux_df['Type'].map(type_labels)


fig = px.ecdf(
    aux_df,
    x='Difference',
    color='Type',
    labels={'Difference': 'Difference between Random/Masked and Consensus Branch Lengths'},
    color_discrete_sequence=['#F49E17', '#18974C']
)

fig.add_vline(x=1, line_dash='dash', line_color='black', annotation_text='Difference = 1', annotation_position='top right')

# Hatch 1 - 2
fig.add_shape(
    type="rect",
    x0=-1,
    y0=0,
    x1=1,
    y1=1.1,
    fillcolor="rgba(0, 0, 0, 0.2)",
    line=dict(color="black", width=0),
)

fig.update_layout(xaxis_title='Difference between Random/Masked and Consensus branch lengths', yaxis_title='Proportion')
# square shape fig

fig.update_layout(width=800, height=400)
fig.write_html("../figs/difference_random_masked_consensus_branch_lengths_distribution.html")
fig.show()
